# Lab Assignment 7 – Grid Search on a Deep Learning Model (Keras + Scikit-learn)
**Course:** CSET343 – AI in Healthcare | **B.Tech, 4th Year, Sem VII**

**Objective:** Use scikit-learn's `GridSearchCV` on a Keras deep-learning model.
**Dataset:** Pima Indians Diabetes (768 patients, 8 clinical features, binary `Outcome`: 1 = diabetic).

**Plan**
1. Data acquisition & loading (Task 1)
2. Preprocessing (Task 2): zero→missing, median imputation, scaling, 70/15/15 split
3. Deep-learning grid search (Task 3): Keras model wrapped for scikit-learn (**SciKeras**), then tune, one group at a time,
   (a) batch size & epochs, (b) learning rate, (c) weight initialisation, (d) activation, (e) dropout, (f) hidden neurons
4. Train the final model with all best values, compare with the untuned baseline, evaluate on the test set

## 0. Setup

In [ ]:
# Run once if needed.  NOTE: the old `keras.wrappers.scikit_learn` was removed; its maintained replacement is SciKeras.
# !pip install -q tensorflow scikeras scikit-learn pandas numpy matplotlib seaborn

import os, time, warnings, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, roc_curve, classification_report)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scikeras.wrappers import KerasClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); keras.utils.set_random_seed(SEED)
sns.set_style("whitegrid")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## Task 1: Data Acquisition and Loading
The loader first looks for a local CSV (e.g. Kaggle's `diabetes.csv`), otherwise it downloads the UCI copy from a public mirror.
To use your own file, place it next to the notebook (`diabetes.csv` or `pima-indians-diabetes.csv`) or set `LOCAL_PATH`.

In [ ]:
COLS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
LOCAL_PATH = None     # e.g. r"C:\data\diabetes.csv"
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

def load_pima():
    paths = [LOCAL_PATH] if LOCAL_PATH else ["diabetes.csv", "pima-indians-diabetes.csv", "pima-indians-diabetes.data.csv"]
    for p in paths:
        if p and os.path.exists(p):
            d = pd.read_csv(p)
            if "Outcome" not in d.columns:             # headerless copy
                d = pd.read_csv(p, header=None, names=COLS)
            print("Loaded local file:", p); return d
    print("Downloading from", URL)
    return pd.read_csv(URL, header=None, names=COLS)

df = load_pima()
print("Shape:", df.shape)
df.head()

In [ ]:
print("Random sample of 8 patients:")
display(df.sample(8, random_state=SEED))
print("\nData types / non-null counts:"); df.info()
print("\nSummary statistics:"); display(df.describe().T.round(2))

### Class balance

In [ ]:
counts = df["Outcome"].value_counts().sort_index()
print(f"Non-diabetic (0): {counts[0]}   Diabetic (1): {counts[1]}   ->  {counts[1]/len(df):.1%} positive")
print(f"Majority-class baseline accuracy: {counts.max()/len(df):.3f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.barplot(x=["No diabetes", "Diabetes"], y=counts.values, palette=["#4C9BE8", "#E8624C"], ax=ax[0])
for i, v in enumerate(counts.values): ax[0].text(i, v + 5, str(v), ha="center", fontweight="bold")
ax[0].set_title("Class balance")
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1], cbar=False, annot_kws={"size": 7})
ax[1].set_title("Correlation matrix")
sns.boxplot(data=df, x="Outcome", y="Glucose", palette=["#4C9BE8", "#E8624C"], ax=ax[2]); ax[2].set_title("Glucose by outcome")
plt.tight_layout(); plt.show()

df.drop(columns="Outcome").hist(bins=25, figsize=(14, 7), color="#4C9BE8"); plt.suptitle("Feature distributions (raw)"); plt.tight_layout(); plt.show()

## Task 2: Data Preprocessing
1. **Impossible zeros → missing.** A value of 0 for Glucose, BloodPressure, SkinThickness, Insulin or BMI is physiologically impossible, so it denotes a missing measurement (`Pregnancies` = 0 is valid and is kept).
2. **Split first (70 / 15 / 15, stratified)**, then fit the imputer (median) and scaler (standardisation) on the **training set only** to avoid data leakage.

In [ ]:
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df_clean = df.copy()
df_clean[ZERO_AS_MISSING] = df_clean[ZERO_AS_MISSING].replace(0, np.nan)
miss = df_clean.isna().sum().to_frame("missing"); miss["%"] = (miss["missing"] / len(df) * 100).round(1)
display(miss[miss["missing"] > 0])

X = df_clean.drop(columns="Outcome"); y = df_clean["Outcome"].values
feature_names = X.columns.tolist()

X_tr_raw, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_va_raw, X_te_raw, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

imputer = SimpleImputer(strategy="median").fit(X_tr_raw)
X_train_i = imputer.transform(X_tr_raw)
scaler = StandardScaler().fit(X_train_i)
prep = lambda A: scaler.transform(imputer.transform(A)).astype("float32")
X_train, X_val, X_test = prep(X_tr_raw), prep(X_va_raw), prep(X_te_raw)

display(pd.DataFrame({"Split": ["Train", "Validation", "Test"],
                      "Patients": [len(y_train), len(y_val), len(y_test)],
                      "Positive %": [f"{y_train.mean():.1%}", f"{y_val.mean():.1%}", f"{y_test.mean():.1%}"]}))
print("Train feature means ~0:", X_train.mean(0).round(2), "\nTrain feature stds  ~1:", X_train.std(0).round(2))

## Task 3: Deep Learning Grid Search
### 3.1 Keras model wrapped for scikit-learn
`build_model` is a function that creates and compiles a Keras network; every argument (`neurons`, `activation`, `init`, `dropout_rate`, `learning_rate`) is a hyper-parameter. SciKeras' `KerasClassifier` turns it into a scikit-learn estimator, so that `GridSearchCV` can use it:

* model arguments are tuned as `model__<arg>` (e.g. `model__neurons`)
* training arguments are tuned directly (`batch_size`, `epochs`)

**Network:** Input(8) → Dense(neurons, activation, init) → Dropout → Dense(8, activation, init) → Dense(1, sigmoid); loss = binary cross-entropy, optimiser = Adam.

In [ ]:
def build_model(meta, neurons=12, activation="relu", init="glorot_uniform", dropout_rate=0.0, learning_rate=0.001):
    n_features = meta["n_features_in_"]
    model = keras.Sequential([
        layers.Input((n_features,)),
        layers.Dense(neurons, activation=activation, kernel_initializer=init),
        layers.Dropout(dropout_rate),
        layers.Dense(8, activation=activation, kernel_initializer=init),
        layers.Dense(1, activation="sigmoid", kernel_initializer=init),
    ])
    model.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=learning_rate), metrics=["accuracy"])
    return model

# Default (un-tuned) hyper-parameters = our baseline
best = {"batch_size": 32, "epochs": 50,
        "model__learning_rate": 0.001, "model__init": "glorot_uniform",
        "model__activation": "relu", "model__dropout_rate": 0.0, "model__neurons": 12}

def make_clf(**params):
    return KerasClassifier(model=build_model, verbose=0, random_state=SEED, **params)

keras.utils.set_random_seed(SEED)
build_model({"n_features_in_": 8}).summary()

### 3.2 Grid-search utilities
Every search uses **stratified 3-fold cross-validation on the training set only** (validation and test sets stay untouched) and scores **accuracy**. Hyper-parameters are tuned in the order requested by the assignment; after each search the winning value is stored in `best` and used by all later searches (the classic *sequential / coordinate* grid search – much cheaper than one huge joint grid).

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
search_log = {}     # step name -> results DataFrame
summary_rows = []

def run_grid(step, grid, scoring="accuracy"):
    base = {k: v for k, v in best.items() if k not in grid}          # everything not being tuned stays at its current best
    gs = GridSearchCV(make_clf(**base), grid, cv=cv, scoring=scoring, n_jobs=1, refit=False, return_train_score=False)
    t0 = time.time(); gs.fit(X_train, y_train)
    res = pd.DataFrame(gs.cv_results_)
    keep = [c for c in res.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score", "rank_test_score"]
    res = res[keep].sort_values("rank_test_score").reset_index(drop=True)
    res.columns = [c.replace("param_model__", "").replace("param_", "") for c in res.columns]
    search_log[step] = res
    old = {k: best[k] for k in grid}
    best.update(gs.best_params_)
    summary_rows.append({"Step": step, "Tuned": ", ".join(k.replace("model__", "") for k in grid),
                         "Best value": ", ".join(f"{v}" for v in gs.best_params_.values()),
                         "CV accuracy": round(gs.best_score_, 4), "Time (s)": round(time.time() - t0)})
    print(f"[{step}] best = {gs.best_params_}  |  CV accuracy = {gs.best_score_:.4f}  ({time.time()-t0:.0f}s)")
    plot_grid(step, res)
    display(res.head(10).round(4))
    return res

def plot_grid(step, res):
    pcols = [c for c in res.columns if c not in ("mean_test_score", "std_test_score", "rank_test_score")]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    if len(pcols) == 1:
        r = res.sort_values(pcols[0]); x = r[pcols[0]].astype(str)
        ax.bar(x, r["mean_test_score"], yerr=r["std_test_score"], color="#4C9BE8", capsize=4)
        lo = max(0, r["mean_test_score"].min() - 0.05); ax.set_ylim(lo, min(1, r["mean_test_score"].max() + 0.04))
        ax.set_xlabel(pcols[0]); ax.set_ylabel("CV accuracy")
    else:
        piv = res.pivot(index=pcols[0], columns=pcols[1], values="mean_test_score")
        sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"Grid search: {step}"); plt.tight_layout(); plt.show()

### 3.3 Tune batch size and training epochs
Batch size = how many samples per gradient update; epochs = passes over the training set. They interact, so they are tuned together.

In [ ]:
_ = run_grid("1. Batch size & epochs", {"batch_size": [8, 16, 32, 64, 128], "epochs": [10, 50, 100]})

### 3.4 Tune the learning rate
Too small → slow/under-trained; too large → unstable/diverging loss.

In [ ]:
_ = run_grid("2. Learning rate", {"model__learning_rate": [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]})

### 3.5 Tune network weight initialisation
The starting weights affect convergence speed and whether the network gets stuck. Candidates: random uniform/normal, Glorot (Xavier), He and zeros (a deliberately poor baseline).

In [ ]:
_ = run_grid("3. Weight initialisation", {"model__init": ["random_uniform", "random_normal", "glorot_uniform", "glorot_normal", "he_uniform", "he_normal", "zeros"]})

### 3.6 Tune the activation function
Controls non-linearity and gradient flow in the hidden layers (the output layer stays sigmoid for binary probability).

In [ ]:
_ = run_grid("4. Activation function", {"model__activation": ["relu", "tanh", "sigmoid", "softplus", "softsign", "elu", "linear"]})

### 3.7 Tune dropout regularisation
Dropout randomly switches off a fraction of hidden neurons during training to reduce over-fitting (0.0 = no dropout).

In [ ]:
_ = run_grid("5. Dropout rate", {"model__dropout_rate": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]})

### 3.8 Tune the number of neurons in the hidden layer
Too few neurons → under-fitting; too many → over-fitting on a small (768-row) dataset.

In [ ]:
_ = run_grid("6. Hidden neurons", {"model__neurons": [4, 8, 12, 16, 24, 32, 64, 128]})

### 3.9 Summary of the sequential search

In [ ]:
summary = pd.DataFrame(summary_rows); display(summary)
print("\nFinal combined hyper-parameters:")
for k, v in best.items(): print(f"  {k:<22} = {v}")

In [ ]:
# OPTIONAL: small joint grid around the sequential optimum to check for interactions (set True to run; ~5-10 min on CPU)
RUN_JOINT = False
if RUN_JOINT:
    joint = {"model__neurons":       sorted({max(4, best["model__neurons"] // 2), best["model__neurons"], best["model__neurons"] * 2}),
             "model__dropout_rate":  sorted({0.0, best["model__dropout_rate"], 0.3}),
             "model__learning_rate": sorted({best["model__learning_rate"] / 2, best["model__learning_rate"], best["model__learning_rate"] * 2}),
             "epochs":               sorted({max(10, best["epochs"] // 2), best["epochs"], best["epochs"] * 2})}
    _ = run_grid("7. Joint refinement", joint)
    print("Updated best:", best)

### 3.10 Final model vs. untuned baseline
Both models are trained on the 70 % training set and evaluated on the held-out **validation** and **test** sets (never used during the search).

In [ ]:
def fit_eval(name, params, n_seeds=5):
    # average over several seeds: small datasets give noisy single-run numbers
    rows, probs, last = [], [], None
    for s in range(n_seeds):
        keras.utils.set_random_seed(SEED + s)
        clf = KerasClassifier(model=build_model, verbose=0, random_state=SEED + s, **params).fit(X_train, y_train)
        row = {}
        for split, Xs, ys in [("val", X_val, y_val), ("test", X_test, y_test)]:
            p = clf.predict_proba(Xs)[:, 1]; pr = (p >= 0.5).astype(int)
            row.update({f"{split}_Accuracy": accuracy_score(ys, pr), f"{split}_Precision": precision_score(ys, pr, zero_division=0),
                        f"{split}_Recall": recall_score(ys, pr), f"{split}_F1": f1_score(ys, pr), f"{split}_AUROC": roc_auc_score(ys, p)})
            if split == "test": probs.append(p)
        rows.append(row); last = clf
    m = pd.DataFrame(rows).agg(["mean", "std"]).T
    return m, np.mean(probs, axis=0), last

baseline_params = {"batch_size": 32, "epochs": 50, "model__learning_rate": 0.001, "model__init": "glorot_uniform",
                   "model__activation": "relu", "model__dropout_rate": 0.0, "model__neurons": 12}
m_base, p_base, clf_base = fit_eval("Baseline", baseline_params)
m_tuned, p_tuned, clf_tuned = fit_eval("Tuned", best)

cmp = pd.DataFrame({"Baseline (mean)": m_base["mean"], "Baseline (std)": m_base["std"],
                    "Tuned (mean)": m_tuned["mean"], "Tuned (std)": m_tuned["std"]}).round(4)
cmp["Δ (tuned-base)"] = (cmp["Tuned (mean)"] - cmp["Baseline (mean)"]).round(4)
display(cmp)

In [ ]:
# Training curves of the tuned model + ROC + confusion matrix (averaged test probabilities of 5 seeds)
pred_t = (p_tuned >= 0.5).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
h = clf_tuned.history_
ax[0].plot(h["loss"], label="loss"); ax[0].plot(h["accuracy"], label="accuracy")
ax[0].set_title("Tuned model – training curves"); ax[0].set_xlabel("epoch"); ax[0].legend()
for name, p in [("Baseline", p_base), ("Tuned", p_tuned)]:
    fpr, tpr, _ = roc_curve(y_test, p); ax[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
ax[1].plot([0, 1], [0, 1], "k--", alpha=.5); ax[1].set_title("ROC (test)"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend()
sns.heatmap(confusion_matrix(y_test, pred_t), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2],
            xticklabels=["Pred No", "Pred Yes"], yticklabels=["True No", "True Yes"]); ax[2].set_title("Tuned model – confusion matrix (test)")
plt.tight_layout(); plt.show()
print(classification_report(y_test, pred_t, target_names=["No diabetes", "Diabetes"], digits=3))

## Observations & Conclusion
* **Approach:** A Keras network was wrapped with SciKeras so that scikit-learn's `GridSearchCV` (stratified 3-fold CV, accuracy) could tune batch size, epochs, learning rate, weight initialisation, activation, dropout and hidden-layer width. The search is *sequential* – each group is tuned with the earlier winners fixed – which needs ~40 grid points per run rather than the several hundred thousand of a full joint grid (an optional joint refinement is included).
* **What typically happens (check against your plots/tables):**
  * *Batch size & epochs:* too few epochs under-fit; very many epochs can start to over-fit; small batches give noisier but more frequent updates.
  * *Learning rate:* extremes (≤1e-4 or ≥0.05) usually score worse – too slow vs. unstable.
  * *Initialisation:* `zeros` is the clear loser (all neurons in a layer stay symmetric); Glorot/He-type initialisers are typically best and most stable.
  * *Activation:* ReLU/tanh/ELU/softplus normally beat `sigmoid` (vanishing gradients) and `linear` (no non-linearity).
  * *Dropout:* a small/moderate rate often helps a little; a high rate removes too much capacity for such a tiny network.
  * *Neurons:* beyond a modest width, accuracy plateaus – the dataset is small and noisy.
* **Interpreting the gain:** Pima Indians Diabetes is small (768 rows) and noisy, so accuracy tops out around 75-80 % for most models and differences of ~1 % between grid points are within CV noise (see `std_test_score`). The final comparison therefore averages 5 random seeds; judge improvement by the mean *and* the spread.
* **Medical note:** in diabetes screening, **recall** (missing no diabetic patient) is often more important than raw accuracy – the decision threshold can be lowered to trade precision for recall. This model is for learning purposes, not clinical use.
* **Caveats:** the search maximised CV accuracy on the training split only; validation/test sets were used just once for the final report, so the reported numbers are an unbiased estimate of generalisation.